<a href="https://colab.research.google.com/github/dkalchenko01/UA-Locations-NER-Transformers/blob/roberta-large/notebooks/03_xlm_roberta_large.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XLM-RoBERTa-large with Weighted Layer Pooling

- Pretrained multilingual Transformer encoder
- Learnable weighted pooling of the last 4 hidden layers
- Token-level BIO classification
- Validation v2

## Environment Setup

In [11]:
%pip install -q transformers datasets accelerate sentencepiece seqeval wandb kagglehub

In [99]:
import os
import sys
import ast
import json
import random
import gc
import shutil
import platform
import subprocess
import importlib
from pathlib import Path
from collections import Counter
from importlib.metadata import version
from time import perf_counter
from safetensors.torch import load_file

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import wandb

from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModel, AutoConfig
from transformers import DataCollatorForTokenClassification
from transformers import Trainer, TrainingArguments, TrainerCallback, EarlyStoppingCallback, set_seed
from transformers.modeling_outputs import TokenClassifierOutput
from seqeval.metrics import f1_score, precision_score, recall_score

ENVIRONMENT = "colab" if "google.colab" in sys.modules else "kaggle"

if ENVIRONMENT == "colab":
    from google.colab import userdata, drive
else:
    from kaggle_secrets import UserSecretsClient

In [13]:
root = Path("/content" if ENVIRONMENT == "colab" else "/kaggle/working")

repo = "UA-Locations-NER-Transformers"
branch = "roberta-large"
repo_dir = root / repo

if not repo_dir.exists():
    subprocess.run(["git", "clone", "-b", branch, f"https://github.com/dkalchenko01/{repo}.git", str(repo_dir)], check=True)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("Repository:", repo_dir)

Repository: /content/UA-Locations-NER-Transformers


Repository synchronization

In [89]:
subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
importlib.invalidate_caches()
for name in list(sys.modules):
    if name.startswith("src."):
        importlib.reload(sys.modules[name])

git_revision = subprocess.check_output(["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True).strip()
print("Git revision:", git_revision)

Git revision: 02b547cf140c685ae6bdd4f9e0712a1c4eab6b17


Project imports

In [90]:
from src.data_io import get_data_dir, load_manifest, load_data_splits
from src.experiment_tracking import start_run, log_evaluation, finish_run, log_kaggle_result
from src.dataset import create_datasets, load_or_tokenize_datasets, prepare_labeling_dataset
from src.modeling import TransformerNER
from src.optimization import build_optimizer
from src.metrics import comp_metric
from src.preprocessing import bio_to_spans
from src.evaluation import build_compute_metrics
from src.callbacks import LabelingSampleCallback, CheckpointBackupCallback

splits_dir = repo_dir / "data" / "splits"

print("Modules imported successfully")
print("Validation manifest exists:", (splits_dir / "validation_manifest_v2.parquet").exists())

Modules imported successfully
Validation manifest exists: True


## External Services (W&B + Drive)

In [16]:
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"
os.environ["WANDB_DISABLE_CODE"] = "true"
os.environ["WANDB_CONSOLE"] = "off"

if ENVIRONMENT == "colab":
    wandb.login(key=userdata.get("WANDB_API_KEY"), relogin=True)
else:
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"), relogin=True)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: No netrc file found, creating one.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /root/.netrc
wandb: Currently logged in as: daldoshyna (daldoshyna-kyiv-school-of-economics) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


In [17]:
if ENVIRONMENT == "colab":
    drive.mount("/content/drive")
    checkpoint_backup_root = Path("/content/drive/MyDrive/UA-Locations-NER-Transformers/xlm_roberta_large")
else:
    checkpoint_backup_root = root / "checkpoints/xlm_roberta_large"

checkpoint_backup_root.mkdir(parents=True, exist_ok=True)
print("Checkpoint backup root:", checkpoint_backup_root)

Mounted at /content/drive
Checkpoint backup root: /content/drive/MyDrive/UA-Locations-NER-Transformers/xlm_roberta_large


In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Transformers:", version("transformers"))
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

PyTorch: 2.14.1+cpu
Transformers: 5.22.1
Device: cpu


## Control Configuration

XLM-RoBERTa-large with learnable weighted pooling of the last 4 hidden layers and token-level BIO classification.

In [97]:
config = {
    "model": "xlm_roberta_large",
    "model_name": "FacebookAI/xlm-roberta-large",
    "architecture": "weighted_layer_pooling",
    "tokenization_version": "v1",
    "pooling_layers": 4,
    "dropout": 0.1,
    "label_to_id": {"O": 0, "B-LOC": 1, "I-LOC": 2},
    "seed": 17,
    "sources": ["uk", "ru"],
    "max_length": 512,
    "batch_size": 4,
    "gradient_accumulation_steps": 8,
    "trainable_encoder_layers": 8,
    "encoder_lr": 1e-5,
    "head_lr": 1e-4,
    "layer_lr_decay": 0.9,
    "optimizer": "adamw_torch",
    "weight_decay": 0.01,
    "scheduler": "cosine",
    "warmup_ratio": 0.1,
    "max_grad_norm": 1.0,
    "epochs": 5,
    "early_stopping_patience": 2,
    "environment": ENVIRONMENT,
    "validation_version": "v2",
    "manifest_name": "validation_manifest_v2",
    "metric_version": "kaggle_original_regex_v1",
    "wandb_entity": "daldoshyna-kyiv-school-of-economics",
    "wandb_project": "ua-location-ner-transformers",
    "decoding": "viterbi_lines",
    "postprocessing_version": "raw",
    "reannotated": False
}

set_seed(config["seed"])

Run directories

In [20]:
run_id = config["model"] + "_" + pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")

run_dir = root / "outputs" / run_id
backup_dir = checkpoint_backup_root / run_id
manifest_path = splits_dir / f"{config['manifest_name']}.parquet"

run_dir.mkdir(parents=True, exist_ok=False)
backup_dir.mkdir(parents=True, exist_ok=True)

print("Run:", run_id)
print("Local directory:", run_dir)
print("Backup directory:", backup_dir)

Run: xlm_roberta_large_20261126T144714088112Z
Local directory: /content/outputs/xlm_roberta_large_20261126T144714088112Z
Backup directory: /content/drive/MyDrive/UA-Locations-NER-Transformers/xlm_roberta_large/xlm_roberta_large_20261126T144714088112Z


Provenance

In [21]:
provenance = {
    "run_id": run_id,
    "seed": config["seed"],
    "git_revision": git_revision,
    "git_branch": branch,
    "python": platform.python_version(),
    "packages": {name: version(name) for name in [
        "torch", "transformers", "tokenizers", "datasets",
        "accelerate", "numpy", "pandas", "pyarrow", "wandb"
    ]},
    "device": str(device),
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
}

## Data Loading

Reusing the fixed validation v2 splits from Assignment 3.

In [22]:
data_dir = get_data_dir(ENVIRONMENT)
manifest = load_manifest(splits_dir, name=config["manifest_name"])

display(pd.crosstab(manifest["source"], manifest["split"]))

100%|██████████| 811M/811M [00:11<00:00, 75.0MB/s]

Extracting files...


split,holdout,train,val
source,,,
ru,24743,100000,24626
uk,100757,805983,100741


Train, validation and holdout splits

In [44]:
splits = load_data_splits(
    ENVIRONMENT, splits_dir,
    sources=config["sources"],
    manifest_name=config["manifest_name"],
    reannotated=config["reannotated"]
)

for name, df in splits.items():
    print(f"{name}: {len(df):,} texts")

uk_train: 805,983 texts
uk_val: 100,741 texts
uk_holdout: 100,757 texts
ru_train: 100,000 texts
ru_val: 24,626 texts
ru_holdout: 24,743 texts


Test and labeling sample

In [24]:
test_df = pd.read_csv(data_dir / "test.csv")
labeling_sample = pd.read_csv(data_dir / "labeling_sample.csv")
print(f"Test: {len(test_df):,} texts")
print(f"Labeling sample: {len(labeling_sample):,} texts")
display(labeling_sample.head(3))

Test: 477 texts
Labeling sample: 26 texts


,text,locations,loc_markers
0,-261.310,[],[]
1,"Проспект Бандери стоїть, як і пів міста… довго...",['Проспект Бандери'],"[(0, 16)]"
2,☢️Радіаційний фон (мкР/год) по Київщині станом...,"['Київщині', 'Чорнобиль', 'Київ', 'Тетерів', '...","[(31, 39), (68, 77), (83, 87), (93, 100), (105..."


## Tokenizer Loading

Using the original pretrained XLM-RoBERTa tokenizer with character-level offset mapping for BIO label alignment.

Load pretrained tokenizer

In [25]:
tokenizer = AutoTokenizer.from_pretrained(config["model_name"], use_fast=True)

print("Tokenizer:", type(tokenizer).__name__)
print("Vocabulary size:", len(tokenizer))
print("Maximum sequence length:", tokenizer.model_max_length)
print("Fast tokenizer:", tokenizer.is_fast)
print("Special tokens:", tokenizer.special_tokens_map)

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Tokenizer: XLMRobertaTokenizer
Vocabulary size: 250002
Maximum sequence length: 512
Fast tokenizer: True
Special tokens: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}


## Dataset & Batching

Batched tokenization, BIO label alignment and dynamic padding using Hugging Face.

In [38]:
datasets = create_datasets(splits, config["sources"])
datasets

DatasetDict({
    train: Dataset({
        features: ['row_id', 'text', 'loc_markers', 'org_markers', 'per_markers', 'is_valid', 'source', 'doc_id', 'sent_id', 'text_id'],
        num_rows: 905983
    })
    val: Dataset({
        features: ['row_id', 'text', 'loc_markers', 'org_markers', 'per_markers', 'is_valid', 'source', 'doc_id', 'sent_id', 'text_id'],
        num_rows: 125367
    })
    holdout: Dataset({
        features: ['row_id', 'text', 'loc_markers', 'org_markers', 'per_markers', 'is_valid', 'source', 'doc_id', 'sent_id', 'text_id'],
        num_rows: 125500
    })
})

Tokenization & Caching

In [80]:
cache_root = checkpoint_backup_root.parent / "tokenized_cache"
tokenizer_name = config["model_name"].split("/")[-1]
sources = "_".join(config["sources"])
annotation = "reannotated" if config["reannotated"] else "original"

cache_name = f"{tokenizer_name}_{sources}_{annotation}_{config['manifest_name']}_{config['max_length']}_{config['tokenization_version']}"
cache_path = cache_root / cache_name

tokenized_datasets = load_or_tokenize_datasets(datasets, tokenizer, config["label_to_id"], config["max_length"], cache_path)
tokenized_datasets

Map:   0%|          | 0/905983 [00:00<?, ? examples/s]

train: 1,744 alignment failures


Filter:   0%|          | 0/905983 [00:00<?, ? examples/s]

Map:   0%|          | 0/125367 [00:00<?, ? examples/s]

val: 0 alignment failures


Map:   0%|          | 0/125500 [00:00<?, ? examples/s]

holdout: 0 alignment failures


Saving the dataset (0/3 shards):   0%|          | 0/904239 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/125367 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/125500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['row_id', 'text', 'loc_markers', 'org_markers', 'per_markers', 'is_valid', 'source', 'doc_id', 'sent_id', 'text_id', 'input_ids', 'attention_mask', 'offset_mapping', 'labels', 'valid'],
        num_rows: 904239
    })
    val: Dataset({
        features: ['row_id', 'text', 'loc_markers', 'org_markers', 'per_markers', 'is_valid', 'source', 'doc_id', 'sent_id', 'text_id', 'input_ids', 'attention_mask', 'offset_mapping', 'labels', 'valid'],
        num_rows: 125367
    })
    holdout: Dataset({
        features: ['row_id', 'text', 'loc_markers', 'org_markers', 'per_markers', 'is_valid', 'source', 'doc_id', 'sent_id', 'text_id', 'input_ids', 'attention_mask', 'offset_mapping', 'labels', 'valid'],
        num_rows: 125500
    })
})

Dynamic padding

In [40]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True, label_pad_token_id=-100)

Labeling sample tokenization

In [73]:
tokenized_labeling = prepare_labeling_dataset(labeling_sample, tokenizer, config["label_to_id"], config["max_length"])

Map:   0%|          | 0/26 [00:00<?, ? examples/s]

## Model Architecture

Pretrained XLM-RoBERTa encoder with learnable weighted layer pooling, dropout and a token-level BIO classification head.

In [59]:
model = TransformerNER(
    model_name=config["model_name"],
    num_labels=len(config["label_to_id"]),
    pooling_layers=config["pooling_layers"],
    dropout=config["dropout"]
).to(device)
model.freeze_encoder(config["trainable_encoder_layers"])

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [60]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable share: {trainable_params / total_params:.2%}")

Total parameters: 559,893,511
Trainable parameters: 100,772,871
Trainable share: 18.00%


## Training

### Training Arguments

In [74]:
training_args = TrainingArguments(
    output_dir=str(run_dir / "checkpoints"),
    num_train_epochs=config["epochs"],
    per_device_train_batch_size=config["batch_size"],
    per_device_eval_batch_size=config["batch_size"],
    gradient_accumulation_steps=config["gradient_accumulation_steps"],
    learning_rate=config["encoder_lr"],
    weight_decay=config["weight_decay"],
    optim=config["optimizer"],
    lr_scheduler_type=config["scheduler"],
    warmup_steps=config["warmup_ratio"],
    max_grad_norm=config["max_grad_norm"],
    eval_strategy="epoch",
    save_strategy="epoch",
    batch_eval_metrics=True,
    include_for_metrics=["inputs"],
    logging_strategy="steps",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="uk_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="wandb",
    run_name=run_id,
    seed=config["seed"],
    remove_unused_columns=True,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported()
)

### Evaluation Metrics

In [75]:
compute_metrics = build_compute_metrics(tokenized_datasets["val"], config["label_to_id"])

### Trainer Initialization

In [84]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["val"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stopping_patience"])]
)

### Optimizer Initialization

In [85]:
trainer.optimizer = build_optimizer(trainer, config)

### Callbacks

In [86]:
trainer.add_callback(LabelingSampleCallback(trainer, tokenized_labeling, config["label_to_id"]))
trainer.add_callback(CheckpointBackupCallback(backup_dir))

### W&B Run Initialization

In [95]:
run = start_run(
    model=config["model"],
    config=config,
    name=run_id,
    validation_version=config["validation_version"],
    metric_version=config["metric_version"],
    postprocessing_version=config["postprocessing_version"],
    entity=config["wandb_entity"],
    project=config["wandb_project"],
    repo_dir=repo_dir,
    split_manifest_path=manifest_path,
    provenance=provenance
)

### Model Training

In [ ]:
train_result = trainer.train()

## Checkpoint Selection

In [98]:
checkpoints = sorted(backup_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))

for checkpoint in checkpoints:
    print(checkpoint.name)

print("Automatic best:", Path(trainer.state.best_model_checkpoint).name)
print("Best UK F1:", trainer.state.best_metric)

TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'NoneType'

In [ ]:
selected_checkpoint = None

automatic_best = Path(trainer.state.best_model_checkpoint).name
selected_checkpoint = selected_checkpoint or automatic_best
selected_checkpoint_path = backup_dir / selected_checkpoint

print("Selected checkpoint:", selected_checkpoint_path)

### Load Selected Checkpoint

In [ ]:
weights = load_file(str(selected_checkpoint_path / "model.safetensors"))
trainer.model.load_state_dict(weights)
trainer.model.eval()
del weights

## Final Evaluation

### Holdout Evaluation

In [ ]:
trainer.remove_callback(LabelingSampleCallback)
trainer.remove_callback(EarlyStoppingCallback)
trainer.compute_metrics = build_compute_metrics(tokenized_datasets["holdout"], config["label_to_id"])
holdout_metrics = trainer.evaluate(tokenized_datasets["holdout"], metric_key_prefix="holdout")

### Labeling Sample Evaluation

In [ ]:
trainer.compute_metrics = build_compute_metrics(tokenized_labeling, config["label_to_id"])
labeling_metrics = trainer.evaluate(tokenized_labeling, metric_key_prefix="labeling")

## Saving Results

In [ ]:
results = {
    "automatic_best_checkpoint": automatic_best,
    "best_uk_f1": trainer.state.best_metric,
    "selected_checkpoint": selected_checkpoint,
    "holdout": holdout_metrics,
    "labeling": labeling_metrics
}

for name, content in [("config", config), ("provenance", provenance), ("metrics", results), ("training_history", trainer.state.log_history)]:
    with open(backup_dir / f"{name}.json", "w") as f:
        json.dump(content, f, indent=4, ensure_ascii=False, default=str)

## Inference & Post-processing

### Test Tokenization

### Test Predictions

### Post-processing

### Submission

### Finish W&B Run

In [ ]:
finish_run(
    run,
    summary={**holdout_metrics, **labeling_metrics, "selected_checkpoint": selected_checkpoint},
    best_score=trainer.state.best_metric,
    selection_metric="uk_f1",
    checkpoint_path=backup_dir / automatic_best,
    upload_checkpoint=False
)